## 3. Affichage des fréquences de retard dans un plan construit par  grâce aux catégories Performance, Causes et Contexte - AFM

L'AFM permet de combiner des groupes de variables de natures différentes. On définit trois groupes :
- Performance : variables numériques de retard,
- Causes : pourcentages de retard par cause,
- Contexte : variables qualitatives (service, gares).

On vise à visualiser les fréquences de retard dans un plan qui tient compte simultanément de ces trois dimensions.

In [ ]:
groups = {
    'Performance': [
        'avg_duration', 'planned_trains', 'cancelled_trains', 'dep_late_trains', 
        'dep_avg_late', 'arr_late_trains', 'arr_avg_late'
    ],
    'Causes': [
        'pct_external', 'pct_infra', 'pct_traffic', 
        'pct_rolling', 'pct_station', 'pct_passengers'
    ],
    'Contexte': [
        'service', 'dep_station', 'arr_station'
    ]
}

cols_num = groups['Performance'] + groups['Causes']
cols_quali = groups['Contexte']

In [ ]:
contexte_dummies = pd.get_dummies(data[groups['Contexte']])
new_contexte_cols = contexte_dummies.columns.tolist()

groups_numeric = {
    'Performance': groups['Performance'],
    'Causes': groups['Causes'],
    'Contexte': new_contexte_cols
}

df_mfa_numeric = pd.concat([data[groups['Performance'] + groups['Causes']], contexte_dummies], axis=1)
df_mfa_numeric = df_mfa_numeric.apply(pd.to_numeric, errors="coerce").fillna(0)

mfa = prince.MFA(n_components=10)
mfa = mfa.fit(df_mfa_numeric, groups=groups_numeric)
mfa.eigenvalues_summary

In [ ]:
explained_variance_ratio = mfa.eigenvalues_summary['% of variance'].str.replace('%', '').astype(float)
n_bars = 10

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scree plot
sns.barplot(x=np.arange(1, n_bars + 1), y=explained_variance_ratio[:n_bars],
            color="coral", ax=axes[0])
axes[0].set_xlabel("Composantes")
axes[0].set_ylabel("Variance expliquée (%)")
axes[0].set_title("Scree plot")
axes[0].spines[["top", "right"]].set_visible(False)

# Cumulative explained variance
cumvar = np.cumsum(explained_variance_ratio)
axes[1].plot(np.arange(1, len(cumvar) + 1), cumvar, color="coral", linewidth=2)
axes[1].axhline(90, color="gray", linestyle="--", linewidth=1, label="90%")
axes[1].axvline(3, color="steelblue", linestyle="--", linewidth=1, label="3 composantes")
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance expliquée cumulative (%)")
axes[1].set_title("Variance expliquée")
axes[1].legend()
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Variance expliquée par les 3 premières componsantes : {cumvar[2]:.2f}%")

In [ ]:
col_contribs = mfa.column_contributions_

group_results = {}

for group_name, original_cols in groups.items():
    mask = col_contribs.index.to_series().apply(
        lambda x: any(str(c) in str(x) for c in original_cols)
    )
    group_results[group_name] = col_contribs[mask].sum()
df_group_contribs = pd.DataFrame(group_results).T

df_group_contribs.iloc[:, :3].plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title("Importance des groupes de variables (MFA)", fontsize=14)
plt.ylabel("Contribution cumulée")
plt.xticks(rotation=0)
plt.legend(["Dimension 0", "Dimension 1", "Dimension 2"])
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
cols_quali = ['service', 'dep_station', 'arr_station']
pairs = [(0, 1), (0, 2), (1, 2)]
cols_num = groups['Performance'] + groups['Causes']
palette_retard = {'Jamais':'green', 'Rarement':'yellow', 'Régulièrement':'orange', 'Souvent': 'red'}

fig, axes = plt.subplots(1, 3, figsize=(20, 8))
coords=mfa.row_coordinates(df_mfa_numeric)

for i, (dim_x, dim_y) in enumerate(pairs):
    ax_proj = axes[i]
    sns.scatterplot(
        x=coords[dim_x], 
        y=coords[dim_y], 
        hue=retard_col, 
        palette=palette_retard,
        hue_order=['Jamais', 'Rarement', 'Régulièrement', 'Souvent'],
        alpha=0.5, 
        s=30, 
        ax=ax_proj
    )
    ax_proj.set_title(f"Projection : Plan {dim_x} x {dim_y}")
    ax_proj.set_xlabel(f"Dim {dim_x} ({var_x})")
    ax_proj.set_ylabel(f"Dim {dim_y} ({var_y})")
    ax_proj.axhline(0, color='grey', linestyle='--', alpha=0.5)
    ax_proj.axvline(0, color='grey', linestyle='--', alpha=0.5)
    ax_proj.legend(title='Fréquence Retard', loc='upper right', fontsize='small')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.suptitle("Retads des trajets dans le plan MFA", fontsize=20, fontweight='bold')
plt.show()

La pertinence de ce jeu de donnée est qu'il est exploitable en terme temporel, depuis le début nous quantifions la notion de retard, mais aussi en terme spatial. Il y a t'il des gares, des trajets plus impactés. Cette partie de graph veut identifier cette "localité" de retard.